# SPATIAL INTELLIGENCE - PART 1

In [29]:
# This cell is not needed if you have pip installed topologicpy
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

## 1. Import the needed libraries

In [30]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## 2. Check the TopologicPy Version

In [31]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This tutorial requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.18) is EQUAL TO the latest version available on PyPI.


## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [32]:
renderer = "vscode"

## 4. Import the OBJ file

In [33]:
objects = Topology.ByOBJPath(r"C:\Users\olgap\OneDrive\IAAC_GraphML_2026\Chloe\Chloe_RhinoGeometry.obj")
# print("Objects is a list")
print(objects)


## 6. Show the geometry

In [34]:
Topology.Show(objects,
              faceColor=[210,210,250],
            #   faceOpacity=1,
              edgeColor="white",
              edgeWidth=3,
              faceOpacity=0.3,
              showVertices=False,
              backgroundColor="white",
              width=800,
              height=600,
              renderer = renderer)

In [35]:
Topology.Show(objects,
              edgeColor=[100,100,200],   # visible color, not white
              edgeWidth=2,
              showFaces=False,           # skip faces entirely
              showVertices=False,
              backgroundColor="white",
              width=800,
              height=600,
              renderer=renderer)

In [36]:
shell = CellComplex.ExternalBoundary(cellComplex)

Topology.Show(shell,
              faceColor=[100, 150, 220],
              faceOpacity=0.3,
              edgeColor=[50, 50, 150],
              edgeWidth=2,
              showVertices=False,
              backgroundColor="white",
              width=800,
              height=600,
              renderer=renderer)

In [37]:
# The imported geometry is a Cluster of faces, not cells
# We need to create a CellComplex which will automatically detect and create cells from the faces

print("Creating CellComplex from faces...")
faces = Topology.Faces(objects[0])
print(f"Number of faces: {len(faces)}")

# Create a CellComplex - this will automatically detect closed volumes (rooms) from the faces
cellComplex = CellComplex.ByFaces(faces, tolerance=0.0001)

# Get all cells (rooms)
cells = Topology.Cells(cellComplex)
print(f"Number of cells (rooms) created: {len(cells)}")

# Find the corridor - it's the cell with the most adjacent cells (highest connectivity)
print("\nAnalyzing cells to find the corridor...")

# Calculate adjacencies for each cell
cell_info = []
for i, cell in enumerate(cells):
    # Count adjacent cells (cells that share a face with this cell)
    adjacent_count = 0
    for j, other_cell in enumerate(cells):
        if i != j:
            # Check if they share faces
            shared_faces = Topology.SharedFaces(cell, other_cell)
            if len(shared_faces) > 0:
                adjacent_count += 1
    
    volume = Cell.Volume(cell)
    cell_info.append((i, volume, adjacent_count, cell))

# Sort by number of adjacent cells (corridor should have most connections)
cell_info.sort(key=lambda x: x[2], reverse=True)

# The corridor is the cell with the most adjacent cells
corridor_idx, corridor_vol, corridor_adjacencies, corridor = cell_info[0]
print(f"Corridor identified: Cell {corridor_idx}")
print(f"  - Volume: {corridor_vol:.2f}")
print(f"  - Adjacent to {corridor_adjacencies} cells")

# Create custom graph with star topology
# ONLY include corridor and rooms directly adjacent to it
corridor_center = Topology.CenterOfMass(corridor)
graph_vertices = [corridor_center]
graph_edges = []

# Only add rooms that are directly adjacent to the corridor
for i, vol, adj, cell in cell_info:
    if i != corridor_idx:  # Skip the corridor itself
        # Check if this room is adjacent to the corridor
        shared_faces = Topology.SharedFaces(cell, corridor)
        if len(shared_faces) > 0:
            # This room connects to the corridor - add it to the graph
            room_center = Topology.CenterOfMass(cell)
            graph_vertices.append(room_center)
            
            # Create edge from room to corridor
            edge = Edge.ByVertices([room_center, corridor_center])
            graph_edges.append(edge)

# Build the star graph
g1 = Graph.ByVerticesEdges(graph_vertices, graph_edges)

print(f"\nStar graph created:")
print(f"  - Vertices: {len(graph_vertices)} (1 corridor + {len(graph_edges)} connected rooms)")
print(f"  - Edges: {len(graph_edges)} (all rooms connect to corridor)")

Creating CellComplex from faces...
Number of faces: 1170
Number of cells (rooms) created: 195

Analyzing cells to find the corridor...
Corridor identified: Cell 2
  - Volume: 3600.00
  - Adjacent to 14 cells

Star graph created:
  - Vertices: 15 (1 corridor + 14 connected rooms)
  - Edges: 14 (all rooms connect to corridor)


In [38]:
# Show graph with building geometry (faces invisible, light edges)
Topology.Show(objects, g1,
              vertexSize=15,
              vertexColor="red",
              edgeWidth=5,
              edgeColor="blue",
              faceOpacity=0.3,  # Completely invisible faces
              showEdges=True,  # Show building edges as wireframe
              width=800,
              height=600,
              backgroundColor="white",
              renderer=renderer)

In [39]:
# Alternative: Show ONLY the graph without building geometry
Graph.Show(g1,
           vertexSize=20,
           vertexColor="red",
           edgeWidth=5,
           edgeColor="blue",
           backgroundColor="white",
           width=800,
           height=600,
           renderer=renderer)

In [40]:
# Create complete graph for ALL rooms across all floors
# Manually place vertices at the center of each room
print("Creating complete graph with all rooms and connections...")

# Create vertices at the center of each room
graph_vertices_all = []
cell_to_vertex_map = {}  # Map cell index to vertex for edge creation

for i, cell in enumerate(cells):
    # Place vertex at the center of mass of the room
    center = Topology.CenterOfMass(cell)
    graph_vertices_all.append(center)
    cell_to_vertex_map[i] = center

print(f"Created {len(graph_vertices_all)} vertices at room centers")

# Create edges between adjacent rooms (rooms that share faces)
graph_edges_all = []
for i, cell1 in enumerate(cells):
    for j, cell2 in enumerate(cells):
        if i < j:  # Avoid duplicates and self-connections
            # Check if they share faces
            shared_faces = Topology.SharedFaces(cell1, cell2)
            if len(shared_faces) > 0:
                # Create edge between their center vertices
                edge = Edge.ByVertices([cell_to_vertex_map[i], cell_to_vertex_map[j]])
                graph_edges_all.append(edge)

print(f"Created {len(graph_edges_all)} edges between adjacent rooms")

# Build the complete graph
g_full = Graph.ByVerticesEdges(graph_vertices_all, graph_edges_all)

print(f"\nComplete graph created:")
print(f"  - Vertices (all rooms): {len(graph_vertices_all)}")
print(f"  - Edges (all connections): {len(graph_edges_all)}")

Creating complete graph with all rooms and connections...
Created 195 vertices at room centers
Created 512 edges between adjacent rooms

Complete graph created:
  - Vertices (all rooms): 195
  - Edges (all connections): 512


## 7. Create Complete Graph for All Floors
Create a graph that includes all rooms across all floors, with edges representing adjacencies.

In [41]:
# Show complete graph for all floors (no building geometry)
Graph.Show(g_full,
           vertexSize=8,
           vertexColor="red",
           edgeWidth=2,
           edgeColor="blue",
           backgroundColor="white",
           width=800,
           height=600,
           renderer=renderer)